# Autism Behavior Detection — 7 Improvements Pipeline

This notebook runs the improvement pipeline on a **pre-trained 93.8% model**.

### Before running:
1. Set **Runtime → Change runtime type → T4 GPU**
2. Upload to Google Drive (`MyDrive/autism_project/`):
   - `dataset/` folder (with `arm_flapping/`, `head_banging/`, `spinning/`, `normal/` subfolders)
   - `autism_mobilenet_gru_phase1.h5` (trained model)
   - `label_encoder.pkl` (fitted encoder)

## Cell 1 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tensorflow==2.13.0 opencv-python>=4.8.0 scikit-learn>=1.3.0 \
    matplotlib>=3.7.0 seaborn>=0.12.0 "numpy>=1.24.0,<2.0.0"

## Cell 2 — Copy Data to Local SSD

Reading from Drive's FUSE mount is 5-10x slower than local SSD. We copy everything to `/content/` first.

In [ ]:
import shutil, os

# ============================================================
# CONFIGURE THESE PATHS TO MATCH YOUR GOOGLE DRIVE STRUCTURE
# ============================================================
DRIVE_PROJECT = "/content/drive/MyDrive/autism_project"
LOCAL_DIR     = "/content/autism_run"
# ============================================================

os.makedirs(LOCAL_DIR, exist_ok=True)

# Copy dataset
dataset_src = os.path.join(DRIVE_PROJECT, "dataset")
dataset_dst = os.path.join(LOCAL_DIR, "dataset")
if os.path.exists(dataset_src) and not os.path.exists(dataset_dst):
    print("Copying dataset to local SSD (this may take a few minutes)...")
    shutil.copytree(dataset_src, dataset_dst)
    print("Done.")
elif os.path.exists(dataset_dst):
    print("Dataset already on local SSD.")
else:
    print(f"ERROR: Dataset not found at {dataset_src}")

# Copy model files
for f in ["autism_mobilenet_gru_phase1.h5",
          "autism_mobilenet_gru_phase2.h5",
          "label_encoder.pkl"]:
    src = os.path.join(DRIVE_PROJECT, f)
    dst = os.path.join(LOCAL_DIR, f)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"Copied: {f}")

print("\nLocal files:")
for f in os.listdir(LOCAL_DIR):
    full = os.path.join(LOCAL_DIR, f)
    if os.path.isfile(full):
        size_mb = os.path.getsize(full) / 1024 / 1024
        print(f"  {f} ({size_mb:.1f} MB)")
    else:
        print(f"  {f}/")

## Cell 3 — Clone Repository & Configure Paths

In [ ]:
import subprocess, sys

REPO_URL = "https://github.com/YOUR_USERNAME/Autism-video-analysis-.git"  # <-- UPDATE THIS
REPO_DIR = "/content/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

# Add model code to Python path
MODEL_CODE_DIR = os.path.join(REPO_DIR, "model")
if MODEL_CODE_DIR not in sys.path:
    sys.path.insert(0, MODEL_CODE_DIR)

# Change working directory so relative imports work
os.chdir(MODEL_CODE_DIR)

# Override CONFIG to point to Colab paths
from config import CONFIG
CONFIG["dataset_path"] = os.path.join(LOCAL_DIR, "dataset")
CONFIG["output_dir"]   = LOCAL_DIR

print("\nConfig updated:")
print(f"  dataset_path = {CONFIG['dataset_path']}")
print(f"  output_dir   = {CONFIG['output_dir']}")

## Cell 4 — Enable T4 GPU + Mixed Precision

In [ ]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print(f"\u2705 GPU detected: {gpus[0].name}")
    print(f"\u2705 Mixed Precision: {mixed_precision.global_policy().name}")
else:
    print("\u274c No GPU detected!")
    print("Go to: Runtime \u2192 Change runtime type \u2192 T4 GPU")

## Cell 5 — Run the 7-Improvement Pipeline

This cell runs all 7 improvements sequentially. Each one:
1. Loads the baseline model from disk
2. Applies its technique (TTA, Mixup, Label Smoothing, etc.)
3. Evaluates on the test set
4. Saves a checkpoint

**Expected runtime**: ~30-60 minutes on T4 GPU.

In [ ]:
from improve import main
main()

## Cell 6 — Save Results Back to Google Drive

In [ ]:
import glob

DRIVE_OUTPUT = os.path.join(DRIVE_PROJECT, "results")
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

saved = []
for pattern in ["*.h5", "*.png", "*.pkl"]:
    for f in glob.glob(os.path.join(LOCAL_DIR, pattern)):
        dst = os.path.join(DRIVE_OUTPUT, os.path.basename(f))
        shutil.copy2(f, dst)
        size_mb = os.path.getsize(f) / 1024 / 1024
        saved.append((os.path.basename(f), size_mb))
        print(f"  Saved: {os.path.basename(f)} ({size_mb:.1f} MB)")

print(f"\n\u2705 {len(saved)} files saved to: {DRIVE_OUTPUT}")

## Cell 7 — Cleanup (Optional)

Free up Colab disk space by removing the local copy of the dataset.

In [ ]:
# Uncomment to delete local dataset copy and free disk space:
# shutil.rmtree(os.path.join(LOCAL_DIR, "dataset"), ignore_errors=True)
# print("Local dataset deleted to free disk space.")

print("\nDone! Your results are safely stored in Google Drive.")
print(f"Drive path: {DRIVE_OUTPUT}")